# 第24章 异常检测与新奇发现

这个 notebook 用一个教学版光谱特征数据集演示：最近邻距离分数如何做最朴素的异常排序，LOF-like 分数如何强调局部密度差异，简化版 isolation score 如何把容易被隔离的点顶出来，以及为什么高分样本还需要人工复核。

## 学习目标

- 理解异常检测主要是在做候选排序
- 区分全局异常、局部异常和处理伪影
- 比较距离、LOF-like 和 isolation score 的差别
- 把高分样本重新落回人工复核逻辑
- 认识阈值本身也是工作流决策

In [ ]:
from __future__ import annotations

import csv
import math
import platform
import random
from pathlib import Path

DATA_PATH = Path('../../data/small/spectral_anomaly_demo.csv').resolve()

with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = []
    for raw in csv.DictReader(handle):
        rows.append({
            'spectrum_id': raw['spectrum_id'],
            'label': raw['label'],
            'halpha_ew': float(raw['halpha_ew']),
            'caii_k_ew': float(raw['caii_k_ew']),
            'blue_red_ratio': float(raw['blue_red_ratio']),
            'snr': float(raw['snr']),
            'reference_flag': raw['reference_flag'],
        })

print(f'Loaded {len(rows)} spectral samples from {DATA_PATH.name}')
flag_counts = {}
for row in rows:
    flag_counts[row['reference_flag']] = flag_counts.get(row['reference_flag'], 0) + 1
print('reference flags:', flag_counts)
print('first row:', rows[0])
print('Python version:', platform.python_version())


## 标准化特征

异常检测同样很依赖尺度，因此先把几个教学特征放到可比较的尺度上。

In [ ]:
feature_names = ['halpha_ew', 'caii_k_ew', 'blue_red_ratio', 'snr']
means = {}
stds = {}
for name in feature_names:
    values = [row[name] for row in rows]
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    means[name] = mean_value
    stds[name] = math.sqrt(variance) or 1.0

standardized_rows = []
for row in rows:
    scaled = {name: (row[name] - means[name]) / stds[name] for name in feature_names}
    standardized_rows.append({**row, **scaled})

print('means:', {name: round(value, 3) for name, value in means.items()})
print('stds:', {name: round(value, 3) for name, value in stds.items()})
print('known unusual ids:', [row['spectrum_id'] for row in standardized_rows if row['reference_flag'] != 'normal'])


## 方法 1：最近邻距离分数

这是最朴素的异常排序方式：看一个点离最近邻居们平均有多远。

In [ ]:
def feature_vector(row):
    return [row[name] for name in feature_names]

def euclidean_distance(left_vector, right_vector):
    return math.sqrt(sum((left - right) ** 2 for left, right in zip(left_vector, right_vector)))

vectors = {row['spectrum_id']: feature_vector(row) for row in standardized_rows}
k_neighbors = 3

mean_neighbor_distance = {}
nearest_neighbors = {}
for row in standardized_rows:
    source_id = row['spectrum_id']
    distances = []
    for other in standardized_rows:
        other_id = other['spectrum_id']
        if other_id == source_id:
            continue
        distances.append((euclidean_distance(vectors[source_id], vectors[other_id]), other_id))
    distances.sort(key=lambda item: item[0])
    nearest_neighbors[source_id] = [other_id for _, other_id in distances[:k_neighbors]]
    mean_neighbor_distance[source_id] = sum(distance for distance, _ in distances[:k_neighbors]) / k_neighbors

distance_ranking = sorted(mean_neighbor_distance.items(), key=lambda item: item[1], reverse=True)
print('Top candidates by mean neighbor distance:')
for source_id, score in distance_ranking[:5]:
    row = next(item for item in standardized_rows if item['spectrum_id'] == source_id)
    print(source_id, round(score, 3), row['reference_flag'], nearest_neighbors[source_id])


## 方法 2：LOF-like 局部密度分数

这里我们用一个简化版 LOF 逻辑，看看“局部上不合群”的点会不会排到前面。

In [ ]:
k_distance = {}
for source_id in vectors:
    ordered = sorted(
        euclidean_distance(vectors[source_id], vectors[other_id])
        for other_id in vectors
        if other_id != source_id
    )
    k_distance[source_id] = ordered[k_neighbors - 1]

def reachability_distance(left_id, right_id):
    raw_distance = euclidean_distance(vectors[left_id], vectors[right_id])
    return max(k_distance[right_id], raw_distance)

local_reachability_density = {}
for source_id in vectors:
    average_reachability = sum(
        reachability_distance(source_id, neighbor_id)
        for neighbor_id in nearest_neighbors[source_id]
    ) / k_neighbors
    local_reachability_density[source_id] = 1.0 / average_reachability

lof_like_score = {}
for source_id in vectors:
    lof_like_score[source_id] = sum(
        local_reachability_density[neighbor_id] / local_reachability_density[source_id]
        for neighbor_id in nearest_neighbors[source_id]
    ) / k_neighbors

lof_ranking = sorted(lof_like_score.items(), key=lambda item: item[1], reverse=True)
print('Top candidates by LOF-like score:')
for source_id, score in lof_ranking[:5]:
    row = next(item for item in standardized_rows if item['spectrum_id'] == source_id)
    print(source_id, round(score, 3), row['reference_flag'], nearest_neighbors[source_id])


## 方法 3：简化版 isolation score

这个版本模拟 Isolation Forest 的核心直觉：越容易被随机切分单独隔离出来，越像异常候选。

In [ ]:
random.seed(7)

def isolate_depth(candidate_id, subset_ids, depth=0):
    if len(subset_ids) <= 1:
        return depth
    feature_indices = []
    for index, name in enumerate(feature_names):
        values = [vectors[source_id][index] for source_id in subset_ids]
        if max(values) - min(values) > 1e-12:
            feature_indices.append(index)
    if not feature_indices:
        return depth
    feature_index = random.choice(feature_indices)
    values = [vectors[source_id][feature_index] for source_id in subset_ids]
    split_value = random.uniform(min(values), max(values))
    left_ids = [source_id for source_id in subset_ids if vectors[source_id][feature_index] < split_value]
    right_ids = [source_id for source_id in subset_ids if vectors[source_id][feature_index] >= split_value]
    next_subset = left_ids if candidate_id in left_ids else right_ids
    if len(next_subset) == len(subset_ids):
        return depth + 1
    return isolate_depth(candidate_id, next_subset, depth + 1)

all_ids = [row['spectrum_id'] for row in standardized_rows]
average_isolation_depth = {}
for source_id in all_ids:
    depths = [isolate_depth(source_id, all_ids) for _ in range(200)]
    average_isolation_depth[source_id] = sum(depths) / len(depths)

isolation_score = {source_id: 1.0 / depth for source_id, depth in average_isolation_depth.items()}
isolation_ranking = sorted(isolation_score.items(), key=lambda item: item[1], reverse=True)
print('Top candidates by simplified isolation score:')
for source_id, score in isolation_ranking[:5]:
    row = next(item for item in standardized_rows if item['spectrum_id'] == source_id)
    print(source_id, round(score, 3), row['reference_flag'], round(average_isolation_depth[source_id], 3))


## 跨方法候选清单与人工复核视角

这里我们把三种排序方式的前几名合并成一个复核队列，强调异常检测的真正落点是“优先看谁”。

In [ ]:
top_candidates = []
for ranking in [distance_ranking, lof_ranking, isolation_ranking]:
    for source_id, _ in ranking[:3]:
        if source_id not in top_candidates:
            top_candidates.append(source_id)

print('Cross-method anomaly review list:')
for source_id in top_candidates:
    row = next(item for item in standardized_rows if item['spectrum_id'] == source_id)
    print({
        'spectrum_id': source_id,
        'label': row['label'],
        'reference_flag': row['reference_flag'],
        'distance_score': round(mean_neighbor_distance[source_id], 3),
        'lof_like': round(lof_like_score[source_id], 3),
        'isolation_score': round(isolation_score[source_id], 3),
    })

print('Manual-review note:')
print('  CR01 ranks very high, but its reference flag says artifact; this is exactly why high anomaly score is not equal to new physics.')
print('  EM01 and WD01 stay near the top across methods and are stronger follow-up candidates.')


## 一个简单阈值示意

阈值本身也是工作流决策：设得太低，复核量会暴涨；设得太高，又可能漏掉真正有趣的样本。

In [ ]:
distance_threshold = distance_ranking[2][1]
review_queue = [source_id for source_id, score in distance_ranking if score >= distance_threshold]
print('Example review threshold (distance score >= top-3 cutoff):', round(distance_threshold, 3))
print('Review queue:', review_queue)


In [ ]:
try:
    from sklearn.ensemble import IsolationForest
    from sklearn.neighbors import LocalOutlierFactor
except ModuleNotFoundError:
    print('scikit-learn 未安装；已跳过标准库异常检测示例。')
else:
    feature_matrix = [feature_vector(row) for row in standardized_rows]
    lof = LocalOutlierFactor(n_neighbors=3)
    lof_predictions = lof.fit_predict(feature_matrix)
    lof_scores = [-value for value in lof.negative_outlier_factor_]
    isolation_forest = IsolationForest(random_state=7, contamination=0.16)
    isolation_forest.fit(feature_matrix)
    forest_scores = [-value for value in isolation_forest.score_samples(feature_matrix)]
    print('sklearn LOF top 5:')
    for source_id, score in sorted(zip(all_ids, lof_scores), key=lambda item: item[1], reverse=True)[:5]:
        print(source_id, round(score, 3))
    print('sklearn IsolationForest top 5:')
    for source_id, score in sorted(zip(all_ids, forest_scores), key=lambda item: item[1], reverse=True)[:5]:
        print(source_id, round(score, 3))


## Algorithm Understanding Bridge

异常检测的关键不是“最高分就是发现”，而是知道每种分数在问什么问题。这一段导出一份候选复核说明，帮助学生把算法分数接到人工 triage。

In [ ]:
RESULTS_DIR = Path('results/part3_algorithm_understanding')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

triage_rows = []
for source_id in top_candidates:
    row = next(item for item in standardized_rows if item['spectrum_id'] == source_id)
    trigger_methods = []
    if source_id in [item[0] for item in distance_ranking[:3]]:
        trigger_methods.append('distance')
    if source_id in [item[0] for item in lof_ranking[:3]]:
        trigger_methods.append('lof-like')
    if source_id in [item[0] for item in isolation_ranking[:3]]:
        trigger_methods.append('isolation')
    if row['reference_flag'] == 'artifact':
        triage = 'bad data / artifact check first'
    elif row['reference_flag'] == 'unusual':
        triage = 'scientifically interesting candidate after data-quality check'
    else:
        triage = 'boundary or ordinary sample to verify'
    triage_rows.append({
        'spectrum_id': source_id,
        'reference_flag': row['reference_flag'],
        'trigger': '+'.join(trigger_methods),
        'distance_score': mean_neighbor_distance[source_id],
        'lof_like': lof_like_score[source_id],
        'isolation_score': isolation_score[source_id],
        'triage': triage,
    })

triage_lines = [
    f"- {row['spectrum_id']}: trigger={row['trigger']}, flag={row['reference_flag']}, "
    f"distance={row['distance_score']:.3f}, lof={row['lof_like']:.3f}, "
    f"isolation={row['isolation_score']:.3f}, triage={row['triage']}"
    for row in triage_rows
]

anomaly_bridge = f'''# Ch24 Algorithm Understanding Bridge

## Distance-based Anomaly

- Question asked: is this sample far from its nearest neighbors?
- Score used here: mean distance to the {k_neighbors} nearest neighbors.
- Failure mode: sparse but ordinary boundary regions can look anomalous.

## LOF-like Local Density

- Question asked: is this sample in a lower-density pocket than its neighbors?
- Score intuition: compare the point's local reachability density with neighbor densities.
- Failure mode: score depends strongly on k and local sampling density.

## Isolation-style Score

- Question asked: is this sample easy to isolate with random feature splits?
- Score intuition: shorter average isolation path means a stronger anomaly candidate.
- Failure mode: random splits can emphasize feature ranges rather than physical novelty.

## Autoencoder Reconstruction Error

- Question asked: does a compressed representation reconstruct this sample poorly?
- Bridge to later deep learning: useful for high-dimensional spectra/images, but reconstruction failure can also mean preprocessing mismatch.

## Review Triage

{chr(10).join(triage_lines)}

## Claim Boundary

Anomaly scores support candidate ranking and review prioritization. They do not by themselves support a discovery claim.
'''

output_path = RESULTS_DIR / 'ch24_algorithm_understanding.md'
output_path.write_text(anomaly_bridge, encoding='utf-8')
print('wrote', output_path)
print('triage rows:', len(triage_rows))


## 小结

这个教学例子说明：异常检测最可靠的角色不是自动宣判，而是给出一张更聪明的复核队列。不同方法顶出来的对象会部分重合、部分不同，这正是它们看待“异常”的角度不同。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_rows': len(rows),
    'top_distance_candidate': distance_ranking[0][0],
    'top_lof_candidate': lof_ranking[0][0],
    'top_isolation_candidate': isolation_ranking[0][0],
    'review_queue': review_queue,
    'python_version': platform.python_version(),
}

print('Anomaly snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')
